In [ ]:
import os
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_path = os.path.join("runs",timestamp)
writer = SummaryWriter(run_path)
best_model_path = os.path.join(run_path,"bestmodel.pth")

# 加载数据

In [42]:
data_path = r"../../data/kaggle/house-prices-advanced-regression-techniques"
df_train = pd.read_csv(f"{data_path}/train.csv")
df_test = pd.read_csv(f"{data_path}/test.csv")
df_train_ids = df_train["Id"]
df_test_ids = df_test["Id"]
df_train.drop(columns=["Id"],inplace=True)
df_test.drop(columns=["Id"],inplace=True)
df_train.shape, df_test.shape

((1460, 80), (1459, 79))

## 数据处理导入

In [43]:
def data_process(df: pd.DataFrame) -> pd.DataFrame:
    numcols = df.select_dtypes(include='number').columns
    df[numcols] = df[numcols].apply(lambda c: (c-c.mean())/(c.std()+1e-8))
    df[numcols].fillna(0.0)
    df = pd.get_dummies(df,dummy_na=True,dtype=np.float32)
    return df.fillna(0.0)
train_rows = len(df_train)
test_rows = len(df_test)
df_alldata = pd.concat([df_train.drop(columns="SalePrice"), df_test], axis=0)
df_alldata = data_process(df_alldata)

df_train_X = df_alldata[:train_rows]
df_test_X = df_alldata[train_rows:]
df_train_y = df_train["SalePrice"]

train_X = torch.tensor(df_train_X.to_numpy(), dtype=torch.float32)
test_X = torch.tensor(df_test_X.to_numpy(), dtype=torch.float32)
train_y = torch.tensor(df_train_y.to_numpy(), dtype=torch.float32)

assert train_X.size(0) == train_y.size(0)
assert train_X.size(1) == test_X.size(1)

train_X


tensor([[ 0.0673, -0.1844, -0.2178,  ...,  1.0000,  0.0000,  0.0000],
        [-0.8735,  0.4581, -0.0720,  ...,  1.0000,  0.0000,  0.0000],
        [ 0.0673, -0.0559,  0.1372,  ...,  1.0000,  0.0000,  0.0000],
        ...,
        [ 0.3025, -0.1416, -0.1428,  ...,  1.0000,  0.0000,  0.0000],
        [-0.8735, -0.0559, -0.0572,  ...,  1.0000,  0.0000,  0.0000],
        [-0.8735,  0.2439, -0.0293,  ...,  1.0000,  0.0000,  0.0000]])

## Model

In [44]:

class HousePriceModel(nn.Module):
    def __init__(self, input_dims, output_dims):
        super().__init__()
        self.input_dims = input_dims
        self.output_dims = output_dims
        self.hidden_dims = 1024
        self.net = nn.Sequential(
            nn.Linear(self.input_dims, self.hidden_dims),
            nn.ReLU(),
            nn.Linear(self.hidden_dims, self.hidden_dims),
            nn.ReLU(),
            nn.Linear(self.hidden_dims, self.hidden_dims),
            nn.ReLU(),
            nn.Linear(self.hidden_dims, self.hidden_dims),
            nn.ReLU(),
            nn.Linear(self.hidden_dims, self.hidden_dims),
            nn.ReLU(),
            nn.Linear(self.hidden_dims, self.hidden_dims),
            nn.ReLU(),
            nn.Linear(self.hidden_dims, self.output_dims)
        )
    def forward(self, X):
        return self.net(X)
    
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

## Model and DataLoader

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
net = HousePriceModel(train_X.size(1), 1)
net.to(device)
net.apply(init_weights)

# y norm
NORM_LOG = "norm_log"
NORM_MINMAX = "norm_minmax"
NORM_MEANSTD = "norm_meanstd"

y_min = train_y.min()
y_max = train_y.max()
y_mean = train_y.mean()
y_std = train_y.std() + 1e-8
y_min.to(device)
y_max.to(device)
y_mean.to(device)
y_std.to(device)

norm_method = NORM_MINMAX

if norm_method == NORM_LOG:
    train_y = train_y.log1p()
elif norm_method == NORM_MINMAX:
    train_y = (train_y - y_min) / (y_max - y_min)
else:
    train_y = (train_y - y_mean) / y_std


num_batches = 20
train_X = train_X.to(device)
train_y = train_y.to(device)
test_X = test_X.to(device)
y_mean.to(device)
y_std.to(device)
dataset = TensorDataset(train_X, train_y)
dataloder = DataLoader(dataset, batch_size=train_rows//20, shuffle=True)

dummy_input = torch.randn(train_X.shape).to(device)
writer.add_graph(net, dummy_input)


cuda


In [46]:
num_epochs = 10000
lr = 1e-3
weight_decay = 1e-2

optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.01, patience=10)
lossfunc = nn.MSELoss(reduction='mean')

def rmsdloss(pred_y: torch.Tensor, batch_y: torch.Tensor):
    return ((pred_y - batch_y) ** 2).mean().sqrt().item()

minLoss = 1E10

net.train()
for epoch in range(num_epochs):
    accumloss = 0
    evalbatch = random.randint(0,19)
    for batch_idx, (batch_X, batch_y) in enumerate(dataloder):
        if evalbatch == batch_idx:
            with torch.no_grad():
                net.eval()
                l = rmsdloss(net(batch_X), batch_y)
                print(epoch, l)
                writer.add_scalar('loss', l, epoch)
                if l < minLoss:
                    minLoss = l
                    torch.save(net.state_dict(), best_model_path)
                # scheduler.step(l)
                    
        else:
            net.train()
            net.zero_grad()
            pred_y = net(batch_X).squeeze(1)
            loss = lossfunc(pred_y, batch_y)
            loss.backward()
            optimizer.step()

writer.close()
print(minLoss)


0 0.20864900946617126
1 0.09955451637506485
2 0.13143251836299896
3 0.15030033886432648
4 0.17201563715934753
5 0.13772380352020264
6 0.17282575368881226
7 0.12412378191947937
8 0.17293404042720795
9 0.15289515256881714
10 0.1426474004983902
11 0.1807975172996521
12 0.12199682742357254
13 0.163545161485672
14 0.13067185878753662
15 0.1952427178621292
16 0.12769058346748352
17 0.12765322625637054
18 0.14113683998584747
19 0.15548062324523926
20 0.18330661952495575
21 0.14467459917068481
22 0.11650336533784866
23 0.15778346359729767
24 0.12410752475261688
25 0.22708135843276978
26 0.16429266333580017
27 0.17904260754585266
28 0.15842172503471375
29 0.14098094403743744
30 0.197183758020401
31 0.14440202713012695
32 0.16795550286769867
33 0.13174857199192047
34 0.1911596804857254
35 0.14847299456596375
36 0.2040216028690338
37 0.1510816216468811
38 0.15823565423488617
39 0.16372241079807281
40 0.15731294453144073
41 0.18899022042751312
42 0.14534200727939606
43 0.15784819424152374
44 0.134

In [ ]:
statc_dict = torch.load(best_model_path)
net.load_state_dict(statc_dict)
net.eval()
with torch.inference_mode():
    test_y = net(test_X).squeeze(1)
    if norm_method == NORM_LOG:
        test_y = test_y.expm1()
    elif norm_method == NORM_MINMAX:
        test_y = test_y * (y_max - y_min) + y_min
    else:
        test_y = test_y * y_std + y_mean
    test_prices = test_y.cpu().detach().numpy()
    df_result = pd.DataFrame({
        "Id": df_test_ids,
        "SalePrice": test_prices
    })
    df_result.to_csv("submission.csv", index=False)